In [ ]:
# General notebook settings
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Multi-Year CfD Settlement: Indexation and Vintage Stacking

The [CfD/PPA settlement example](cfd-ppa-settlement.ipynb) settles a support contract for
a single year. Real schemes run for 15-20 years, which brings in two effects that the
single-year recipe misses:

1. **Strike indexation** - the strike escalates with inflation. UK CfD strikes, for
   instance, are CPI-indexed annually, so the guaranteed price is
   $S_y = S_0\,(1+\pi)^{y-y_0}$.
2. **Vintage stacking** - the support cost in any calendar year is the sum over several
   cohorts ("vintages"), each commissioned in a different auction round, each locked at
   its own strike for its own contract window.

Both stay pure post-processing: solve a normal LOPF once, read the prices, and settle the
contracts as exogenous cash flows. Only the multi-year bookkeeping is new. See
[issue #1475](https://github.com/PyPSA/PyPSA/issues/1475) for the wider discussion.

!!! note "Exogenous price path"
    For simplicity the multi-year market price here is the base-year shape scaled by a
    constant drift. A higher-fidelity study would re-solve each year with the evolving
    fleet (e.g. via [multi-period optimisation](multi-investment-optimisation.ipynb)) and
    feed those prices into the same settlement step.

## Base Year

We solve a single bidding zone over one representative year to obtain an hourly price
shape and the wind asset's realised (post-curtailment) output. This is the only
optimisation in the notebook; the multi-year settlement below reuses these results.

In [ ]:
Y0 = 2030
n_hours = 8760
snapshots = pd.date_range(f"{Y0}-01-01", periods=n_hours, freq="h")
hour = np.arange(n_hours)

# Seasonal (winter-peaking) plus diurnal demand.
seasonal = np.cos(2 * np.pi * hour / n_hours)  # +1 mid-winter, -1 mid-summer
diurnal = np.sin(2 * np.pi * (hour - 9) / 24)
load = pd.Series(320 + 90 * seasonal + 110 * diurnal, index=snapshots)

# Wind capacity factor: winter-high, with synoptic and diurnal variation.
cf = np.clip(
    0.40
    + 0.18 * seasonal
    + 0.30 * np.sin(2 * np.pi * hour / 73)
    + 0.10 * np.sin(2 * np.pi * hour / 11),
    0.02,
    1.0,
)
cf = pd.Series(cf, index=snapshots)

P_NOM_WIND = 480.0

n = pypsa.Network()
n.set_snapshots(snapshots)
n.add("Bus", "market")
for carrier in ["wind", "ccgt_base", "ccgt_mid", "ccgt_high", "ocgt_peak"]:
    n.add("Carrier", carrier)
n.add(
    "Generator",
    "wind",
    bus="market",
    carrier="wind",
    p_nom=P_NOM_WIND,
    marginal_cost=-3.0,
    p_max_pu=cf,
)
n.add(
    "Generator",
    "ccgt_base",
    bus="market",
    carrier="ccgt_base",
    p_nom=250,
    marginal_cost=30.0,
)
n.add(
    "Generator",
    "ccgt_mid",
    bus="market",
    carrier="ccgt_mid",
    p_nom=250,
    marginal_cost=70.0,
)
n.add(
    "Generator",
    "ccgt_high",
    bus="market",
    carrier="ccgt_high",
    p_nom=200,
    marginal_cost=110.0,
)
n.add(
    "Generator",
    "ocgt_peak",
    bus="market",
    carrier="ocgt_peak",
    p_nom=500,
    marginal_cost=180.0,
)
n.add("Load", "demand", bus="market", p_set=load)
n.optimize(solver_name="highs")

In [ ]:
base_price = n.buses_t.marginal_price["market"]
weights = n.snapshot_weightings.generators
# Realised availability per MW of wind, reused for every vintage and year.
realised_cf = n.generators_t.p["wind"] / P_NOM_WIND
energy_per_mw = realised_cf * weights  # MWh per MW, per snapshot
annual_mwh_per_mw = float(energy_per_mw.sum())
capture_base = float((base_price * energy_per_mw).sum() / energy_per_mw.sum())

print(
    f"Time-average price:     {(base_price * weights).sum() / weights.sum():6.1f} EUR/MWh"
)
print(f"Wind capture price:     {capture_base:6.1f} EUR/MWh")
print(f"Full-load hours per MW: {annual_mwh_per_mw:6.0f} MWh/MW/yr")
print(f"Negative-price hours:   {(base_price < 0).sum():6d}")

The capture price sits below the time-average: wind earns less in the hours it produces
most (cannibalisation). The support scheme is what insulates the investor from this.

## 1. Strike Indexation (single asset)

One asset commissioned in $y_0$ is settled each year against a drifting market price
$\lambda_y(t) = \lambda_0(t)\,(1+g)^{y-y_0}$ and a CPI-indexed strike
$S_y = S_0\,(1+\pi)^{y-y_0}$. The CfD top-up is the same one-liner as the single-year
case, $(S_y - \lambda_y)\cdot g_t$, summed per year.

In [ ]:
S0 = 70.0  # EUR/MWh strike in the commissioning year
CPI = 0.02  # strike indexation per year
DRIFT = 0.01  # market price drift per year
CONTRACT_YEARS = 15
CAPACITY_MW = 500.0

years = np.arange(Y0, Y0 + CONTRACT_YEARS)
mwh_per_year = CAPACITY_MW * annual_mwh_per_mw

rows = []
for y in years:
    k = y - Y0
    price_y = base_price * (1 + DRIFT) ** k
    strike_y = S0 * (1 + CPI) ** k
    cfd_eur = float(((strike_y - price_y) * CAPACITY_MW * energy_per_mw).sum())
    capture_y = float((price_y * CAPACITY_MW * energy_per_mw).sum() / mwh_per_year)
    rows.append(
        {
            "year": int(y),
            "strike": round(strike_y, 1),
            "capture_price": round(capture_y, 1),
            "cfd_subsidy_meur": round(cfd_eur / 1e6, 2),
            "achieved_price": round(capture_y + cfd_eur / mwh_per_year, 1),
        }
    )
indexation = pd.DataFrame(rows).set_index("year")
print(
    f"Lifetime CfD subsidy (undiscounted): {indexation['cfd_subsidy_meur'].sum():.0f} MEUR"
)
indexation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    indexation.index, indexation["strike"], "o-", color="C3", label="Indexed strike"
)
ax.plot(
    indexation.index,
    indexation["capture_price"],
    "s-",
    color="C0",
    label="Market capture price",
)
ax.fill_between(
    indexation.index,
    indexation["capture_price"],
    indexation["strike"],
    where=(indexation["strike"] >= indexation["capture_price"]),
    color="C2",
    alpha=0.2,
    label="CfD top-up",
)
ax.set_xlabel("calendar year")
ax.set_ylabel("EUR/MWh")
ax.set_title("Indexed strike vs market capture price")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

With indexation the guaranteed strike outpaces a modestly drifting market, so the annual
top-up widens over the contract: the scheme gets more expensive per MWh as it ages.

## 2. Vintage Stacking (portfolio)

Now stack cohorts. Vintage $v$ (commissioned in year $v$) adds `CAPACITY_MW`, locks its
strike at commissioning $S_v = S_0(1+\pi)^{v-y_0}$ (each auction round clears a little
higher), and is settled for `CONTRACT_YEARS`. In calendar year $Y$ the total support is
the sum over all vintages still inside their contract window:

$$ \text{CfD}_Y = \sum_{v\ \text{active in}\ Y} (S_v - \lambda_Y)\, g_v . $$

In [ ]:
N_VINTAGES = 5

vintages = np.arange(Y0, Y0 + N_VINTAGES)
last_year = Y0 + N_VINTAGES - 1 + CONTRACT_YEARS - 1
calendar_years = np.arange(Y0, last_year + 1)

# Cash-flow matrix: rows = calendar year, columns = vintage cohort (MEUR).
support = pd.DataFrame(0.0, index=calendar_years, columns=[f"v{v}" for v in vintages])
for v in vintages:
    strike_v = S0 * (1 + CPI) ** (v - Y0)
    for y in range(v, v + CONTRACT_YEARS):
        price_y = base_price * (1 + DRIFT) ** (y - Y0)
        cfd_eur = float(((strike_v - price_y) * CAPACITY_MW * energy_per_mw).sum())
        support.loc[y, f"v{v}"] = cfd_eur / 1e6

peak_year = int(support.sum(axis=1).idxmax())
print(f"Total portfolio support (undiscounted): {support.to_numpy().sum():.0f} MEUR")
print(f"Peak year {peak_year}: {support.sum(axis=1).max():.0f} MEUR/yr")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bottom = np.zeros(len(calendar_years))
cmap = plt.get_cmap("viridis", len(vintages))
for i, col in enumerate(support.columns):
    ax.bar(
        calendar_years,
        support[col].to_numpy(),
        bottom=bottom,
        color=cmap(i),
        label=col,
        width=0.85,
    )
    bottom += support[col].to_numpy()
ax.plot(calendar_years, support.sum(axis=1).to_numpy(), "k--", lw=1, label="total")
ax.set_xlabel("calendar year")
ax.set_ylabel("CfD support [MEUR]")
ax.set_title("Stacked CfD support by vintage")
ax.legend(ncol=3, fontsize=7)
plt.tight_layout()
plt.show()

The stack ramps up as cohorts are added, plateaus while they all run, then rolls off as
each contract expires. Indexation tilts later vintages to higher strikes, so the profile
is not symmetric even with identical capacity per round, which matters for budgeting the
scheme's total exposure.

## Notes

- The settlement is the same one line as the single-year example,
  `(strike - price) * dispatch * weights`, evaluated per year and per vintage.
- Curtailment is taken from the base-year solve and held fixed per MW; adding large new
  vintages would in reality depress prices and raise curtailment, which the
  exogenous-price view does not capture.
- All figures are undiscounted nominal EUR. Apply a discount factor to the per-year cash
  flows for an NPV.